In [1]:
# =====================================================================
# 1. IMPORT LIBRARY & LOAD DATASETS
# =====================================================================
import pandas as pd
import numpy as np

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
bids = pd.read_csv('bids.csv')
#======================================================================
# 2. DATA EXPLORATION
# =======================================================================
print(bids.isnull().sum()) #missing values-country
print(bids.nunique()) #unique values
print(bids.duplicated().sum()) #zero duplicate
bids.info()

bid_id            0
bidder_id         0
auction           0
merchandise       0
device            0
time              0
country        8859
ip                0
url               0
dtype: int64
bid_id         7656334
bidder_id         6614
auction          15051
merchandise         10
device            7351
time            776529
country            199
ip             2303991
url            1786351
dtype: int64
0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7656334 entries, 0 to 7656333
Data columns (total 9 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   bid_id       int64 
 1   bidder_id    object
 2   auction      object
 3   merchandise  object
 4   device       object
 5   time         int64 
 6   country      object
 7   ip           object
 8   url          object
dtypes: int64(2), object(7)
memory usage: 525.7+ MB


In [2]:
# =====================================================================
# 3. DATA PREPROCESSING & CONSISTENCY CHECKS
# =====================================================================
# Standardize case formatting to prevent duplicate category codes
for col in ['merchandise', 'device', 'ip', 'url', 'country']:
    bids[col] = bids[col].astype(str).str.lower().str.strip()

# Missing Value
bids['country'] = bids['country'].replace('nan', 'unknown')

# Memory Optimization
bids['bid_id'] = bids['bid_id'].astype('int32')
bids['merchandise'] = bids['merchandise'].astype('category')
bids['country'] = bids['country'].astype('category')
bids['time'] = bids['time'] = bids['time']
#Chronological Sorting
bids = bids.sort_values(by=['bidder_id', 'time'], ascending=True).reset_index(drop=True)

In [3]:
# #comparing unique bidders
# active_bids_bidders = bids['bidder_id'].unique()
# missing_train_bidders = train[~train['bidder_id'].isin(active_bids_bidders)]
# missing_test_bidders = test[~test['bidder_id'].isin(active_bids_bidders)]

# print(f"-> Found {len(missing_train_bidders)} training bidders with 0 bids.")
# print(f"-> Found {len(missing_test_bidders)} testing bidders with 0 bids.")

In [4]:
# #time since last bid to capture bidding patterns
# bids['time_since_last_bid'] = bids.groupby('bidder_id')['time'].diff()
# print(bids['time_since_last_bid'])

In [5]:
# =====================================================================
# 4. TEMPORAL DIFFS & CHANGE FLAGS
# =====================================================================
# Replace NaT (first bid per user) with 0
bids['time_since_last_bid'] = bids.groupby('bidder_id')['time'].diff().fillna(0).clip(lower=0)
# Device / IP change flags (binary: 1 if changed from previous bid)
bids['device_changed'] = (bids.groupby('bidder_id')['device'].shift().bfill() != bids['device']).astype(int)
bids['ip_changed'] = (bids.groupby('bidder_id')['ip'].shift().bfill() != bids['ip']).astype(int)

Three per-bid features are computed using groupby shifts:

a) `time_since_last_bid` (seconds)
`diff()` of time within each bidder group. First bid per user = 0.

Robots bid far faster than humans (milliseconds vs seconds/minutes).

b) `device_changed` (binary)
1 if device differs from previous bid.

Humans use 1-2 devices; bots cycle through hundreds.

c) `ip_changed` (binary)
1 if IP differs from previous bid.

Bots rotate IPs to evade detection; humans typically use 1-2 IPs (home/work).

These three per-timestep features are the direct input to the LSTM. Response time and device/IP diversity are identified in the Kaggle competition literature as the strongest discriminators between human and robot behaviour.

In [6]:
# =====================================================================
# 5. STATIC (AGGREGATED) FEATURES
# =====================================================================
grp = bids.groupby('bidder_id')
static = pd.DataFrame({'bidder_id': grp.size().index}).set_index('bidder_id')
static['total_bids'] = grp.size()
static['total_unique_auctions'] = grp['auction'].nunique()
static['avg_bids_per_auction'] = static['total_bids'] / static['total_unique_auctions']
static['unique_ips'] = grp['ip'].nunique()
static['unique_devices'] = grp['device'].nunique()
static['unique_urls'] = grp['url'].nunique()
static['unique_countries'] = grp['country'].nunique()
static['unique_merchandise'] = grp['merchandise'].nunique()
static['ip_to_bid_ratio'] = static['unique_ips'] / static['total_bids']
static['device_to_bid_ratio'] = static['unique_devices'] / static['total_bids']
static['url_to_bid_ratio'] = static['unique_urls'] / static['total_bids']
tdiff = grp['time_since_last_bid']
static['tdiff_mean_sec'] = tdiff.mean()
static['tdiff_median_sec'] = tdiff.median()
static['tdiff_min_sec'] = tdiff.min()
static['tdiff_max_sec'] = tdiff.max()
static['tdiff_std_sec'] = tdiff.std()
time_min, time_max = grp['time'].min(), grp['time'].max()
static['activity_span_sec'] = (time_max - time_min).clip(lower=1)
static['bids_per_hour'] = static['total_bids'] / (static['activity_span_sec'] / 3600.0)
# Win rate: last bidder per auction
last_bid_idx = bids.groupby('auction')['time'].idxmax()
auction_wins = bids.loc[last_bid_idx].groupby('bidder_id').size()
static['auc_win_rate'] = (auction_wins / static['total_unique_auctions']).fillna(0)
static['tdiff_cv'] = static['tdiff_std_sec'] / static['tdiff_mean_sec'].replace(0, np.nan)
static = static.replace([np.inf, -np.inf], np.nan).fillna(0).reset_index()

Grouped by `bidder_id`, we extracted:

Volume
| Feature | Signal |
|---------|--------|
| `total_bids` | High volume → bot |
| `total_unique_auctions` | Bots enter many auctions |
| `avg_bids_per_auction` | Bots average ~23/auction vs humans ~6 |

Network / Device Diversity
| Feature | Signal |
|---------|--------|
| `unique_ips`, `unique_devices`, `unique_urls`, `unique_countries`, `unique_merchandise` | Bots cycle IPs/devices; humans stay on 1-2 |

Ratio Features
| Feature | Purpose |
|---------|---------|
| `ip_to_bid_ratio`, `device_to_bid_ratio`, `url_to_bid_ratio` | Normalise diversity by activity level |

Temporal Aggregations
| Feature | Signal |
|---------|--------|
| `tdiff_mean`, `median`, `min`, `max`, `std` (seconds) | Bots have much lower and more consistent inter-bid times |

Activity Density
| Feature | Signal |
|---------|--------|
| `activity_span_sec` | Time between first and last bid |
| `bids_per_hour` | Total bids / hours active — bots are active 24/7 with high density |

Proxy Win Rate
| Feature | Calculation | Signal |
|---------|-------------|--------|
| `auc_win_rate` | Auctions where bidder placed last bid / total auctions | Bots win disproportionately more |

Burstiness
| Feature | Calculation | Signal |
|---------|-------------|--------|
| `tdiff_cv` | std / mean (coefficient of variation) | Captures regularity of bidding intervals |

These 20 features encode everything a tree-based model would learn, but as dense numerical vectors suitable for an MLP. They capture the "golden features" identified by top Kaggle solutions. Ratios and CV normalise raw counts so heavy bidders don't dominate.

In [7]:
# =====================================================================
# 6. SEQUENTIAL FEATURES (pad/truncate to 300)
# =====================================================================
MAX_SEQ_LEN = 300
SEQ_COLS = ['time_since_last_bid', 'device_changed', 'ip_changed']
seq_dict = {}
for bidder_id, group in bids.groupby('bidder_id'):
    arr = group[SEQ_COLS].values.astype(np.float64)
    if len(arr) > MAX_SEQ_LEN:
        arr = arr[-MAX_SEQ_LEN:]
    if len(arr) < MAX_SEQ_LEN:
        pad = np.zeros((MAX_SEQ_LEN - len(arr), 3), dtype=np.float64)
        arr = np.vstack([pad, arr])
    seq_dict[bidder_id] = arr

Each bidder's bids are converted to an array of shape (300, 3):

- **Truncation:** If a bidder has >300 bids, we keep the most recent 300. This preserves current behaviour patterns.
- **Pre-padding:** If a bidder has <300 bids, we pad the beginning with zeros. The LSTM learns to ignore padded positions through its gating mechanism.

Sequence Length Selection

| Percentile | Bid Count |
|------------|-----------|
| 50th | 18 |
| 75th | 212 |
| **Selected (300)** | **~80th** |
| 85th | 688 |
| 90th | 1,495 |

Raw sequences preserve the micro-timing that aggregation destroys. An LSTM can learn patterns like "bids spaced exactly 47ms apart" which no aggregation statistic captures. A bot that bids every 50ms for 200 bids then pauses looks like a normal average in aggregations, but the LSTM sees the rigid periodicity. The 300-timestep length captures ~80% of bidders fully while keeping LSTM training memory and time practical.

In [8]:
# =====================================================================
# 7. CATEGORICAL ENCODING (mode per bidder → embeddings)
# =====================================================================
cat_df = pd.DataFrame({
    'bidder_id': bids['bidder_id'].unique()
})
for col in ['country', 'merchandise']:
    mode = bids.groupby('bidder_id')[col].agg(lambda x: x.value_counts().index[0])
    cat_df = cat_df.merge(mode.rename(f'{col}_mode'), on='bidder_id', how='left')
# Device: top-500 + <UNK>
top_devices = bids['device'].value_counts().nlargest(500).index
device_mode = bids.groupby('bidder_id')['device'].agg(lambda x: x.value_counts().index[0])
device_mode = device_mode.apply(lambda x: x if x in top_devices else '<UNK>')
cat_df = cat_df.merge(device_mode.rename('device_mode'), on='bidder_id', how='left')
from sklearn.preprocessing import LabelEncoder
for col in ['country_mode', 'merchandise_mode', 'device_mode']:
    le = LabelEncoder()
    cat_df[f'{col}_encoded'] = le.fit_transform(cat_df[col].astype(str))

For each bidder, we take the mode (most frequent value) of:

| Feature | Unique Values | Mapping |
|---------|---------------|---------|
| `country` | 96 | Label-encoded 0–95 |
| `merchandise` | 10 | Label-encoded 0–9 |
| `device` | 7,351 total → top 500 | Top 500 kept; rest → `<UNK>` (index 0) |

These are label-encoded (0, 1, 2, ...) and fed to `nn.Embedding` layers.

One-hot encoding country (96 dims) or device (500+ dims) creates extremely sparse vectors. Embeddings learn dense, semantic representations — e.g., countries with similar bidding cultures map to nearby vectors. The device cap at 500 prevents a 7,351-vocabulary embedding from being too large to train with only 2,013 training examples. The mode summarises each bidder's typical attribute without requiring per-bid categorical inputs.

In [9]:
# =====================================================================
# 8. ALIGN TO TRAIN/TEST (fill missing bidders with zeros)
# =====================================================================
def align(train_or_test, static_df, seq_dict, cat_df, has_labels=True):
    bidders = train_or_test[['bidder_id']].copy()
    X_static = bidders.merge(static_df, on='bidder_id', how='left').drop(columns='bidder_id').fillna(0).values.astype(np.float64)
    X_seq = np.stack([seq_dict.get(bid, np.zeros((MAX_SEQ_LEN, 3))) for bid in bidders['bidder_id']], axis=0)
    merged_cat = bidders.merge(cat_df, on='bidder_id', how='left')
    cat_arrs = {f'{col}_mode_encoded': merged_cat[f'{col}_mode_encoded'].fillna(0).astype(np.int64).values
                for col in ['country', 'merchandise', 'device']}
    y = train_or_test['outcome'].values.astype(np.float64) if has_labels else None
    return X_static, X_seq, cat_arrs, y
X_static_train, X_seq_train, cat_train, y_train = align(train, static, seq_dict, cat_df)
X_static_test, X_seq_test, cat_test, _ = align(test, static, seq_dict, cat_df, has_labels=False)

29 training bidders and 70 test bidders have zero bids in `bids.csv`. These are not dropped, they receive:

- Zero vectors for static features (20 zeros)
- Zero arrays for sequential features (300 × 3 zeros)
- Zero for categorical encodings

Dropping them would misalign array indices with the label vector (train has 2,013 rows; we must output 2,013 rows). Zero-filled inputs let the model learn a default prediction for unseen/inactive users. In production, a user with no bids is most likely human (since bots are hyperactive), so zeros bias toward the majority class.

In [10]:
# =====================================================================
# 9. SCALING
# =====================================================================
from sklearn.preprocessing import StandardScaler
static_scaler = StandardScaler()
X_static_train = static_scaler.fit_transform(X_static_train)
X_static_test = static_scaler.transform(X_static_test)
seq_scaler = StandardScaler()
flat = X_seq_train.reshape(-1, 3)
mask = ~(flat == 0).all(axis=1)
seq_scaler.fit(flat[mask])
X_seq_train = seq_scaler.transform(X_seq_train.reshape(-1, 3)).reshape(-1, MAX_SEQ_LEN, 3)
X_seq_test = seq_scaler.transform(X_seq_test.reshape(-1, 3)).reshape(-1, MAX_SEQ_LEN, 3)

A `StandardScaler` (z-score) is fitted on the training set only:

- **Static features:** `fit_transform` on train, `transform` on test
- **Sequential features:** fitted only on non-padded timesteps to avoid skewing the scaler towards zero values

Neural networks require inputs centred near 0 with unit variance. Raw features like `total_bids` range from 1 to 515,033, which would cause gradient instability and slow convergence. Fitting on train + transforming test prevents data leakage.

In [11]:
# =====================================================================
# 10. VERIFY SHAPES
# =====================================================================
print(f'MAX_SEQUENCE_LENGTH = {MAX_SEQ_LEN}')
print(f'Static features: {X_static_train.shape[1]}')
print(f'--- TRAIN ---')
print(f'X_static: {X_static_train.shape}, X_seq: {X_seq_train.shape}, y: {y_train.shape}')
print(f'--- TEST ---')
print(f'X_static: {X_static_test.shape}, X_seq: {X_seq_test.shape}')
for k, v in cat_train.items():
    print(f'{k}: {v.shape}')
for k, v in cat_test.items():
    print(f'{k}: {v.shape}')

MAX_SEQUENCE_LENGTH = 300
Static features: 20
--- TRAIN ---
X_static: (2013, 20), X_seq: (2013, 300, 3), y: (2013,)
--- TEST ---
X_static: (4700, 20), X_seq: (4700, 300, 3)
country_mode_encoded: (2013,)
merchandise_mode_encoded: (2013,)
device_mode_encoded: (2013,)
country_mode_encoded: (4700,)
merchandise_mode_encoded: (4700,)
device_mode_encoded: (4700,)


In [ ]:
# # save arrays to test
# np.save('X_static_train.npy', X_static_train)
# np.save('X_seq_train.npy', X_seq_train)
# np.save('y_train.npy', y_train)
# np.save('X_static_test.npy', X_static_test)
# np.save('X_seq_test.npy', X_seq_test)